# CertainRAG benchmark

Runs every benchmark in `evaluation/` on a single T4 GPU. Raw scores are written to `evaluation/results/` and the tables and figures to `evaluation/outputs/`. Every run resumes from its results file, so a cell can be re-run after an interruption without repeating work.

Requirements: GPU accelerator, internet access, and a Hugging Face read token stored as the Kaggle secret `HF_TOKEN` with the LLM-AggreFact terms accepted.

## Setup

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
!git clone -b v0.2 https://github.com/rudranshiverma/CertainRAG.git
%cd CertainRAG
!pip install -q -e ".[bench]" lettucedetect

### Resume from an earlier run

If the output zip of a previous run is attached as an input, its results are restored and finished examples are skipped.

In [ ]:
import glob
import subprocess

for archive in glob.glob("/kaggle/input/*/certainrag_results.zip"):
    subprocess.run(["unzip", "-qo", archive], check=True)
    print(f"restored {archive}")

## Baseline check

Scores three RAGBench answers with each baseline before the long runs. These rows are the first three of the full sample, so they are reused later.

In [ ]:
!python -m evaluation.run_faithfulness --method hhem --datasets ragbench-pubmedqa --splits test --limit 3
!python -m evaluation.run_faithfulness --method lettucedetect-base --datasets ragbench-pubmedqa --splits test --limit 3
!tail -n 3 evaluation/results/faithfulness/hhem/ragbench-pubmedqa_test.jsonl

## Retrieval quality

Scores each RAGBench question against its documents and against documents from a distant subset, then fits Platt scaling on the training split.

In [ ]:
!python -m evaluation.run_retrieval
!python -m evaluation.calibrate_retrieval

## Faithfulness verifiers

RAGTruth and RAGBench answers, then LLM-AggreFact claims with the full test split.

In [ ]:
!python -m evaluation.run_faithfulness --method factcg
!python -m evaluation.run_faithfulness --method factcg --datasets aggrefact --limit 0 --dev-limit 300
!python -m evaluation.run_faithfulness --method minicheck-deberta
!python -m evaluation.run_faithfulness --method minicheck-deberta --datasets aggrefact --limit 0 --dev-limit 300
!python -m evaluation.run_faithfulness --method minicheck-roberta
!python -m evaluation.run_faithfulness --method minicheck-roberta --datasets aggrefact --limit 0 --dev-limit 300

## Baselines

In [ ]:
!python -m evaluation.run_faithfulness --method hhem
!python -m evaluation.run_faithfulness --method lettucedetect-base
!python -m evaluation.run_faithfulness --method lettucedetect-large

## LLM judge

Serves the judge locally with Ollama and scores the RAGTruth QA test split plus 50 answers per RAGBench subset, nested inside the verifier samples.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull phi4-mini

In [ ]:
ragbench = "ragbench-pubmedqa ragbench-covidqa ragbench-techqa ragbench-emanual ragbench-cuad ragbench-delucionqa"
!python -m evaluation.run_faithfulness --method judge --datasets ragtruth-qa {ragbench} --splits test --limit 50
server.terminate()

## Report

In [ ]:
!python -m evaluation.report

from IPython.display import Image, Markdown, display

display(Markdown(open("evaluation/outputs/results.md").read()))
display(Image("evaluation/outputs/pareto.png"))
display(Image("evaluation/outputs/risk_coverage.png"))

## Save results

In [ ]:
!zip -qr /kaggle/working/certainrag_results.zip evaluation/results evaluation/outputs